# Budgerigar：Colab 特征提取

输入是 `Budgerigar_Data.ipynb` 生成并审计通过的 manifest。本 notebook 生成 24 kHz、10 ms hop、100 维因果兼容 log-Mel，以及帧能量和轻量 VAD。

In [ ]:
#@title 1. 更新项目与安装训练依赖
REPO_DIR = '/content/Budgerigar'
from pathlib import Path
import subprocess, sys, importlib
if not Path(REPO_DIR).is_dir():
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/DoctorAwe/Budgerigar.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[train,data]'], check=True)
sys.path.insert(0, REPO_DIR)
for name in [key for key in list(sys.modules) if key == 'budgerigar' or key.startswith('budgerigar.')]:
    del sys.modules[name]
importlib.invalidate_caches()
commit = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('commit:', commit)

In [ ]:
#@title 2. 挂载 Drive 并选择已审计 manifest
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT = Path('/content/drive/MyDrive/Budgerigar')
MANIFEST = WORK_ROOT / 'manifests' / 'cmu_arctic.jsonl'
if not MANIFEST.is_file():
    raise FileNotFoundError('请先完整运行 Budgerigar_Data.ipynb')
print(MANIFEST)

In [ ]:
#@title 3. 检查 GPU/音频依赖
import torch, torchaudio
print('torch:', torch.__version__, 'torchaudio:', torchaudio.__version__, 'cuda:', torch.version.cuda)
if torch.__version__.split('+')[0] != torchaudio.__version__.split('+')[0]:
    raise RuntimeError('torch 与 torchaudio 版本不匹配，请重启 runtime 后重新运行第 1 单元')

In [ ]:
#@title 4. 提取或恢复特征缓存
import json
from budgerigar.features import FeatureConfig, cache_manifest_features
config = FeatureConfig()
feature_manifest, feature_report = cache_manifest_features(
    MANIFEST, WORK_ROOT / 'features' / config.fingerprint,
    WORK_ROOT / 'manifests' / f'cmu_arctic.features.{config.fingerprint}.jsonl', config,
)
report_path = WORK_ROOT / 'reports' / f'feature_report.{config.fingerprint}.json'
report_path.write_text(json.dumps(feature_report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(feature_report, ensure_ascii=False, indent=2))
print('manifest:', feature_manifest, 'report:', report_path)

In [ ]:
#@title 5. 随机抽样可视化与张量检查
import json, random
import matplotlib.pyplot as plt
rows = [json.loads(line) for line in feature_manifest.read_text(encoding='utf-8').splitlines() if line.strip()]
row = random.Random(17).choice(rows)
payload = torch.load(row['feature_path'], map_location='cpu', weights_only=True)
assert payload['log_mel'].ndim == 2 and payload['log_mel'].shape[1] == config.n_mels
assert len(payload['vad']) == len(payload['log_mel'])
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].imshow(payload['log_mel'].T, origin='lower', aspect='auto')
axes[0].set_title(f"{row['id']} | {row['speaker']} | {row['text']}")
axes[1].plot(payload['energy_db'].numpy(), label='energy dB')
axes[1].fill_between(range(len(payload['vad'])), payload['vad'].numpy(), alpha=.25, label='VAD')
axes[1].legend(); plt.show()

In [ ]:
#@title 6. 保存本阶段运行元数据
from budgerigar.experiment import write_run_metadata
metadata = write_run_metadata(
    WORK_ROOT / 'reports' / f'feature_run_metadata.{config.fingerprint}.json',
    feature_manifest, {'feature_fingerprint': config.fingerprint}, repository=REPO_DIR,
)
print(metadata.read_text(encoding='utf-8'))